# BERT Nested Relation Extraction

In [ ]:
import json
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset
from transformers import  AutoModelForSequenceClassification, AutoTokenizer, AutoModel, DataCollatorWithPadding, TrainingArguments, Trainer
import random
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

c:\Users\micha\OneDrive\Documents\Glasgow University\Internship\nested_relation_extraction\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Relations and Special Token Initialisation

In [ ]:
one_arg_rels = [
    'increase',
    'decrease',
    'rna_expression',
    'protein_expression',
    'expression',
    'amplification',
    'deletion',
    'mutation',
    'production',
    'bioactivity',
    'phosphorylation',
    'phosphorylated',
    'dephosphorylation',
    'dephosphorylated',
    'ubiquitination',
    'deubiquitination',
    'methylation',
    'methylated',
    'demethylation',
    'demethylated',
]

two_arg_rels = [
    'effect',
    'correlation',
    #'binding'
]

In [4]:
SPECIAL_TOKENS = [
    '[E1]', '[/E1]',
    '[E2]', '[/E2]',
    '[EFFECT]', '[/EFFECT]',
    '[CORRELATION]', '[/CORRELATION]',
    '[INCREASE]', '[/INCREASE]',
    '[DECREASE]', '[/DECREASE]',

    '[RNA_EXPRESSION]', '[/RNA_EXPRESSION]',
    '[PROTEIN_EXPRESSION]', '[/PROTEIN_EXPRESSION]',
    '[EXPRESSION]', '[/EXPRESSION]',
    '[AMPLIFICATION]', '[/AMPLIFICATION]',
    '[DELETION]', '[/DELETION]',
    '[MUTATION]', '[/MUTATION]',
    '[PRODUCTION]', '[/PRODUCTION]',
    '[BIOACTIVITY]', '[/BIOACTIVITY]',
    '[PHOSPHORYLATION]', '[/PHOSPHORYLATION]',
    '[PHOSPHORYLATED]', '[/PHOSPHORYLATED]',
    '[DEPHOSPHORYLATION]', '[/DEPHOSPHORYLATION]',
    '[DEPHOSPHORYLATED]', '[/DEPHOSPHORYLATED]',
    '[UBIQUITINATION]', '[/UBIQUITINATION]',
    '[DEUBIQUITINATION]', '[/DEUBIQUITINATION]',
    '[METHYLATION]', '[/METHYLATION]',
    '[METHYLATED]', '[/METHYLATED]',
    '[DEMETHYLATION]', '[/DEMETHYLATION]',
    '[DEMETHYLATED]', '[/DEMETHYLATED]',
    #'[BINDING]', '[/BINDING]',

    '[CAUSE]', '[/CAUSE]',
    '[THEME]', '[/THEME]',
    '[VARIABLE]', '[/VARIABLE]',
    '[GENE_PROTEIN]', '[/GENE_PROTEIN]',
    '[BIOMOLECULE]', '[/BIOMOLECULE]',
    #'[PARTNER1]', '[/PARTNER2]',

]

### Tokeniser 

In [5]:
MODEL_NAME = "bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

tokenizer.add_tokens(SPECIAL_TOKENS, True)
model.resize_token_embeddings(len(tokenizer))

c:\Users\micha\OneDrive\Documents\Glasgow University\Internship\nested_relation_extraction\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\micha\.cache\huggingface\hub\models--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 78

Embedding(29054, 768, padding_idx=0)

## BERT Model Training

In [ ]:
all_relation_labels = one_arg_rels + two_arg_rels + ["none"]
label2id = {label: i for i, label in enumerate(all_relation_labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(label2id)
print(num_labels, label2id)

In [ ]:
class RelationDataset(Dataset):
    def __init__(self, examples, tokenizer, label2id, max_length=256):
        self.labels = [label2id[ex["label"]] for ex in examples]
        self.encodings = tokenizer(
            [ex["text"] for ex in examples],
            truncation=True,
            max_length=max_length,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
class MinimalRelationClassifier(nn.Module):
    def __init__(self, encoder, num_labels, dropout=0.1):
        super().__init__()
        self.encoder = encoder
        hidden_size = encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        cls_hidden = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_hidden))

        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
        return {"loss": loss, "logits": logits}

In [ ]:
classifier = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
)
classifier.resize_token_embeddings(len(tokenizer))

In [ ]:
with open("drive/MyDrive/EPSRC Internship/bert_training_examples_small.json") as f:
    examples = json.load(f)

n = len(examples)
indices = list(range(n))
random.seed(42)
random.shuffle(indices)

examples = [examples[i] for i in indices]
train_split = int(0.8 * n)
val_split = int(0.9 * n)

train_examples = examples[:train_split]
val_examples = examples[train_split:val_split]
test_examples = examples[val_split:]

train_dataset = RelationDataset(train_examples, tokenizer, label2id)
val_dataset = RelationDataset(val_examples, tokenizer, label2id)

In [ ]:
with open('train_bert_training_examples_small.json') as f:
    train_examples = json.load(f)

with open('val_bert_training_examples_small.json') as f:
    val_examples = json.load(f)

train_dataset = RelationDataset(train_examples, tokenizer, label2id)
val_dataset = RelationDataset(val_examples, tokenizer, label2id)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = (preds == labels).mean()
    return {"accuracy": accuracy}

training_args = TrainingArguments(
    output_dir="./phase4_minimal",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=classifier,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
trainer.model.eval()
eval_device = trainer.args.device
sample = val_examples[:10]
with torch.no_grad():
    for ex in sample:
        encoded = tokenizer(ex["text"], truncation=True, max_length=256, return_tensors="pt").to(eval_device)
        logits = trainer.model(encoded["input_ids"], encoded["attention_mask"])["logits"]
        pred_label = id2label[logits.argmax(dim=-1).item()]
        print(f"gold={ex['label']:<15} pred={pred_label:<15} text={ex['text'][:80]}...")

In [ ]:
predictions = trainer.predict(val_dataset)
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(axis=1)

report_dict = classification_report(
    y_true,
    y_pred,
    labels=list(range(num_labels)),
    target_names=all_relation_labels,
    digits=4,
    zero_division=0,
    output_dict=True,
)

with open("bert_classification_report.json", "w") as f:
    json.dump(report_dict, f, indent=2)

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(num_labels)), normalize="true")

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=all_relation_labels
)

fig, ax = plt.subplots(figsize=(12, 12))
disp.plot(
    cmap="Blues",
    ax=ax,
    xticks_rotation=90,
    colorbar=False
)

plt.tight_layout()
plt.savefig('cm_v2bert_base_cased.png')
plt.show()